# Healthcare Data Cleaning Notebook

This notebook performs basic data cleaning operations on the raw healthcare CSV files.

**Steps:**
- Load all raw data from `data/datasets/`
- Clean column names (remove spaces, special characters)
- Handle missing values (fill with `'NA'`)
- Drop duplicates
- Export cleaned files to `data/outputs/`

In [1]:
import pandas as pd
import os

input_dir = "../data/datasets/"
output_dir = "../data/outputs/"
os.makedirs(output_dir, exist_ok=True)

## Step 1: Define list of raw CSV files to clean

In [2]:
files = [
    "FactTable.csv",
    "DimPatient.csv",
    "DimPhysician.csv",
    "DimSpeciality.csv",
    "DimHospital.csv",
    "DimPayer.csv",
    "DimCptCode.csv",
    "DimDiagnosisCode.csv",
    "DimDate.csv",
    "DimTransaction.csv"
]

In [3]:
print(len(files))

10


## Step 2: Define a function to clean column names
- Replaces spaces, dashes, and parentheses with underscores

In [4]:
def clean_column(col):
    return col.strip().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")

## Step 3: Loop through each file
- Read, clean, handle nulls, drop duplicates
- Save cleaned file to `data/outputs/`

In [5]:
df = pd.read_csv("../data/datasets/DimPatient.csv")
df.head()

,dimPatientPK,PatientNumber,FirstName,LastName,Email,PatientGender,PatientAge,PatientHeight(in cms),Year of Birth,Month,...,Tobacco,Alcohol,Exercise,Diet,Ethinicity,Zip Codes,State Code,City,State,Region
0,4760825,21376821,Melanie,Greene,melanie.greene@datacourse.com,Female,75,160,1925,7,...,No,Yes,No,No,White,36849,AL,Auburn,Alabama,South
1,4975268,21391171,Neil,Pullman,neil.pullman@datacourse.com,Male,86,174,1914,5,...,Yes,Yes,No,No,Two or More Races,36849,AL,Auburn,Alabama,South
2,5331866,21385004,Kimberly,Mills,kimberly.mills@datacourse.com,Female,65,158,1935,10,...,No,No,No,Yes,White,36849,AL,Auburn,Alabama,South
3,5431278,21365432,Dan,Wilkins,dan.wilkins@datacourse.com,Male,19,180,1981,6,...,No,No,Yes,No,Black/ African American,36849,AL,Auburn,Alabama,South
4,5498935,21387538,Benjamin,Cornish,benjamin.cornish@datacourse.com,Male,28,171,1972,6,...,No,Yes,Yes,No,White,36849,AL,Auburn,Alabama,South


In [6]:
for file in files:
    try:
        df = pd.read_csv(os.path.join(input_dir, file))
        df.columns = [clean_column(c) for c in df.columns]

        df.drop_duplicates(inplace=True)
        df.fillna("NA", inplace=True)

        #  Convert and fix DimDate
        if file == "DimDate.csv":
            for col in df.columns:
                if "date" in col.lower():
                    try:
                        df[col] = pd.to_datetime(df[col], dayfirst=True).dt.strftime('%Y-%m-%d')
                    except Exception as e:
                        print(f"Date conversion failed for {col}: {e}")
            if "Month" in df.columns:
                try:
                    df["Month"] = pd.to_datetime(df["Date"]).dt.month
                except Exception as e:
                    print(f"Month conversion failed: {e}")

        #  Convert and fix FactTable
        if file == "FactTable.csv":
            df.replace({"#NUM!": 0, "#DIV/0!": 0}, inplace=True)
            for col in df.columns:
                if "date" in col.lower():
                    try:
                        df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=True).dt.strftime('%Y-%m-%d')
                    except Exception as e:
                        print(f" Failed to convert {col}: {e}")


        #  Save after conversion
        cleaned_filename = "cleaned_" + file
        df.to_csv(os.path.join(output_dir, cleaned_filename), index=False)
        print(f" Cleaned and saved: {cleaned_filename}")

    except Exception as e:
        print(f" Error processing {file}: {e}")

C:\Users\User\AppData\Local\Temp\ipykernel_13164\4291553262.py:3: DtypeWarning: Columns (14: Adjustment) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(input_dir, file))


 Cleaned and saved: cleaned_FactTable.csv
 Cleaned and saved: cleaned_DimPatient.csv
 Cleaned and saved: cleaned_DimPhysician.csv
 Cleaned and saved: cleaned_DimSpeciality.csv
 Cleaned and saved: cleaned_DimHospital.csv
 Cleaned and saved: cleaned_DimPayer.csv
 Cleaned and saved: cleaned_DimCptCode.csv
 Cleaned and saved: cleaned_DimDiagnosisCode.csv
 Cleaned and saved: cleaned_DimDate.csv
 Cleaned and saved: cleaned_DimTransaction.csv


In [7]:
cleaned_df = pd.read_csv("../data/outputs/cleaned_FactTable.csv")

print(cleaned_df["Adjustment"].dtype)
print(cleaned_df["Adjustment"].head())

int64
0    0
1    0
2    0
3    0
4    0
Name: Adjustment, dtype: int64


In [8]:
cleaned_df["Adjustment"].describe()

count    83999.000000
mean        16.122216
std         52.956571
min          0.000000
25%          0.000000
50%          2.000000
75%          9.000000
max       1602.000000
Name: Adjustment, dtype: float64